In [1]:
from PIL import Image
import os

In [4]:
def resize_image(
        img_path,
        new_width=None,
        new_height=None,
        output_path=None,
        output_format=None
):

    """
    Resize an image to a new size while maintaining its aspect ratio.
    Optionally change the file format of the output image.

    Args:
        dict of:
            path (str): Path to the input image.
            output_path (str): output path.
            new_width (int, optional): new image width. Defaults to None.
            new_height (int, optional): new image height. Defaults to None.
            output_format (str, optional): new file format (e.g., 'JPEG', 'PNG'). Defaults to None.

    Raises:
        ValueError: If both new_width and new_height are not provided.
    """
    try:

        # Open the input image
        with Image.open(img_path) as img:
            # Get original dimensions
            original_width, original_height = img.size

            # Calculate new dimensions to maintain aspect ratio
            if new_width is not None and new_height is None:
                aspect_ratio = original_height / original_width
                new_height = int(new_width * aspect_ratio)
                new_width = int(new_height * aspect_ratio)
                # Resize the image
                resized_img = img.resize((new_width, new_height), Image.LANCZOS)


            # Determine the output format
            if output_format is None:
                output_format = img.format  # Use original format if none specified

            # Ensure the output path has the correct extension
            base, _ = os.path.splitext(output_path)
            output_path = f"{base}.{output_format.lower()}"

            # Save the resized image
            if new_height is not None and new_width is not None:
                resized_img.save(output_path, format=output_format.upper())
            else:
                img.save(output_path, format=output_format.upper())
            print(f"Image resized to {new_width}x{new_height} and saved as {output_path}")

    except FileNotFoundError:
        print(f"Error: File {img_path} not found.")
    except ValueError as e:
        print(f"ValueError: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [6]:
img_name = 'kit-image'
img_format = 'png'

resize_image(
    img_path=f'{img_name}.{img_format}',
    new_width=None,
    new_height=None,
    output_path=img_name,
    output_format='WEBP'
)

Image resized to NonexNone and saved as kit-image.webp


In [ ]:
import zipfile
import os
from PIL import Image
import shutil # For robust temporary directory removal

def convert_zip_image_contents(zip_file_path, target_extension, output_dir=None):
    """
    Opens a .zip file, extracts image files, converts them to the target_extension,
    saves them to a specified output directory, and then zips the output directory.

    Args:
        zip_file_path (str): Path to the input .zip file.
        target_extension (str): The desired new file extension (e.g., 'png', 'jpg', 'webp').
                                 This implies image format conversion.
        output_dir (str, optional): Directory to save the converted files.
                                     If None, a directory named after the zip file
                                     (e.g., 'my_archive_converted') will be created.
    """
    if not os.path.exists(zip_file_path):
        print(f"Error: Zip file not found at {zip_file_path}")
        return

    # Create output directory if it doesn't exist
    if output_dir is None:
        base_name = os.path.splitext(os.path.basename(zip_file_path))[0]
        output_dir = f"{base_name}_converted"
    os.makedirs(output_dir, exist_ok=True)

    print(f"Opening zip file: {zip_file_path}")
    print(f"Converting images to: {target_extension.upper()}")
    print(f"Saving converted images to: {output_dir}")

    temp_extract_dir = os.path.join(output_dir, "temp_zip_extract")
    converted_count = 0

    try:
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            # Extract all contents to a temporary directory
            zip_ref.extractall(path=temp_extract_dir)

            for root, _, files in os.walk(temp_extract_dir):
                for file_name in files:
                    full_path_extracted = os.path.join(root, file_name)
                    relative_path = os.path.relpath(full_path_extracted, temp_extract_dir)

                    # Check if it's an image file
                    if file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.webp', '.tiff')):
                        try:
                            # Get the base filename without original extension
                            base_filename = os.path.splitext(file_name)[0]
                            output_filename = f"{base_filename}.{target_extension.lower()}"
                            output_file_path = os.path.join(output_dir, output_filename)

                            with Image.open(full_path_extracted) as img:
                                img.save(output_file_path, format=target_extension.upper())
                                print(f"Converted '{relative_path}' to '{output_filename}'")
                                converted_count += 1
                        except Image.UnidentifiedImageError:
                            print(f"Warning: Could not identify '{relative_path}' as an image. Skipping.")
                        except Exception as e:
                            print(f"Error converting '{relative_path}': {e}")
                    else:
                        print(f"Skipping non-image file: {relative_path}")

    except zipfile.BadZipFile:
        print(f"Error: '{zip_file_path}' is not a valid zip file.")
    except FileNotFoundError:
        print(f"Error: Zip file not found at {zip_file_path}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    finally:
        # Clean up the temporary extraction directory
        if os.path.exists(temp_extract_dir):
            shutil.rmtree(temp_extract_dir)
            print(f"Cleaned up temporary directory: {temp_extract_dir}")

    print(f"\nConversion complete. {converted_count} image files converted.")

    # Zip the output directory
    if converted_count > 0:
        zip_output_filename = f"{output_dir}.zip"
        try:
            with zipfile.ZipFile(zip_output_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
                for root, _, files in os.walk(output_dir):
                    for file in files:
                        file_path = os.path.join(root, file)
                        # Add file to zip, preserving directory structure relative to output_dir
                        zipf.write(file_path, os.path.relpath(file_path, output_dir))
            print(f"Successfully zipped converted images to: {zip_output_filename}")
        except Exception as e:
            print(f"Error zipping the output directory '{output_dir}': {e}")
    else:
        print("No images were converted, so no output directory to zip.")

In [ ]:
convert_zip_image_contents('./pdf-report.zip', 'WEBP')

Opening zip file: ./pdf-report.zip
Converting images to: WEBP
Saving converted images to: pdf-report_converted
Converted 'Fotos Reporte en PDF/allergies_cowmilk.png' to 'allergies_cowmilk.webp'
Converted 'Fotos Reporte en PDF/spoon_icon.jpg' to 'spoon_icon.webp'
Converted 'Fotos Reporte en PDF/scientist.png' to 'scientist.webp'
Converted 'Fotos Reporte en PDF/food_texture_green.png' to 'food_texture_green.webp'
Converted 'Fotos Reporte en PDF/baby_icon.jpg' to 'baby_icon.webp'
Converted 'Fotos Reporte en PDF/cromosome.png' to 'cromosome.webp'
Converted 'Fotos Reporte en PDF/body_weight.png' to 'body_weight.webp'
Converted 'Fotos Reporte en PDF/gastrointestinal_diarrea.jpg' to 'gastrointestinal_diarrea.webp'
Converted 'Fotos Reporte en PDF/food_texture_red.png' to 'food_texture_red.webp'
Converted 'Fotos Reporte en PDF/shield_icon.jpg' to 'shield_icon.webp'
Converted 'Fotos Reporte en PDF/know_your_child_sugar_intake.jpg' to 'know_your_child_sugar_intake.webp'
Converted 'Fotos Reporte e